In [1]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, callbacks
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, matthews_corrcoef, confusion_matrix
import glob
import os
import joblib
import gc
import warnings
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

np.random.seed(42)
tf.random.set_seed(42)

# СТРИКТНИ НАСТРОЙКИ ЗА ПРЕДПАЗВАНЕ НА RAM ПАМЕТТА
SEQUENCE_LENGTH = 5 
TOP_FEATURES_COUNT = 15
SAMPLES_PER_CLASS_PER_FILE = 2000  

# ============================================
# 1. GPU НАСТРОЙКА (ДИНАМИЧНА ПАМЕТ)
# ============================================
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus: 
            tf.config.experimental.set_memory_growth(gpu, True)
        print(f"[+] Открити {len(gpus)} GPU устройства. Динамичната памет е активирана.")
    except RuntimeError as e:
        print(e)

print("=" * 60)
print("NETWORK INTRUSION DETECTION - ULTIMATE LSTM TRAINING")
print("=" * 60)

# ============================================
# 2. ОТКРИВАНЕ И ОБРАБОТКА НА ФАЙЛОВЕТЕ
# ============================================
current_dir_files = glob.glob("*WorkingHours*.csv")
DATA_FOLDER = "." if len(current_dir_files) >= 4 else ("datasets" if os.path.exists("datasets") else "IP_attack/datasets")
csv_files = glob.glob(os.path.join(DATA_FOLDER, "*.csv"))

if len(csv_files) == 0: 
    raise FileNotFoundError("Не са намерени CSV файлове! Провери директорията.")

def load_stream_and_balance(file_path):
    print(f"Обработка на: {os.path.basename(file_path)} ...")
    chunks = []
    try:
        for chunk in pd.read_csv(file_path, low_memory=False, encoding='latin-1', chunksize=50000):
            chunk.columns = chunk.columns.str.strip()
            chunk = chunk.drop(columns=chunk.columns[chunk.columns == ''], errors='ignore').loc[:, chunk.columns.notna()]
            chunk = chunk.loc[:, ~chunk.columns.duplicated()]
            
            if 'Label' not in chunk.columns:
                continue
                
            chunk['Label'] = chunk['Label'].astype(str).str.strip()
            chunk = chunk[~chunk['Label'].isin(['nan', 'None', '', 'Label'])]
            
            cols_to_drop = ['Flow ID', 'Source IP', 'Src IP', 'Source Port', 'Src Port', 
                            'Destination IP', 'Dst IP', 'Destination Port', 'Dst Port', 'Protocol', 'Timestamp']
            chunk = chunk.drop(columns=cols_to_drop, errors='ignore')
            chunks.append(chunk)
    except Exception as e:
        print(f"[-] Грешка при четене на {os.path.basename(file_path)}: {e}")
        return None

    if not chunks:
        return None
        
    df_file = pd.concat(chunks, ignore_index=True)
    del chunks; gc.collect()
    
    grouped = df_file.groupby('Label')
    balanced_chunks = []
    for name, group in grouped:
        if len(group) >= 10:
            balanced_chunks.append(group.sample(min(SAMPLES_PER_CLASS_PER_FILE, len(group)), random_state=42))
            
    df_balanced = pd.concat(balanced_chunks, ignore_index=True)
    del df_file; gc.collect()
    return df_balanced

# ============================================
# 3. ЗАРЕЖДАНЕ НА ДАННИТЕ С ОН-ЛАЙН ФИЛТРИРАНЕ
# ============================================
print("\n[1] ЗАРЕЖДАНЕ НА ДАННИТЕ С ОН-ЛАЙН ФИЛТРИРАНЕ...")
df_list = []
for f in csv_files:
    df_temp = load_stream_and_balance(f)
    if df_temp is not None:
        df_list.append(df_temp)
        print(f"[✓] Успешно извлечени редове от {os.path.basename(f)}: {len(df_temp)}")

df = pd.concat(df_list, ignore_index=True)
del df_list; gc.collect()

X_raw = df.drop('Label', axis=1)
y_raw = df['Label']

# Изравняване на крайните класове след обединяването
grouped = pd.DataFrame({'Label': y_raw}).groupby('Label')
final_indices = []
for name, group in grouped:
    final_indices.extend(group.sample(min(12000, len(group)), random_state=42).index)

X_raw = X_raw.iloc[final_indices].reset_index(drop=True)
y_raw = y_raw.iloc[final_indices].reset_index(drop=True)
del df; gc.collect()

# -------------------------------------------------------------
# БРОНИРАН ФИКС: Тотално изчистване на крайните матрици
# Преди да ги подадем на RandomForest, премахваме всякакви inf, -inf и NaN
# -------------------------------------------------------------
X_raw = X_raw.replace([np.inf, -np.inf], np.nan).fillna(0)
X_raw = X_raw.astype('float32') # Конвертиране на чисто числовия блок накрая

# Проверка дали случайно не са останали безкрайности след конвертирането
X_raw = X_raw.replace([np.inf, -np.inf], 0).fillna(0)

# ============================================
# 4. FEATURE SELECTION (С RANDOM FOREST)
# ============================================
print(f"\n[2] FEATURE SELECTION: Извличане на топ {TOP_FEATURES_COUNT} характеристики...")
rf = RandomForestClassifier(n_estimators=30, random_state=42, n_jobs=-1)
rf.fit(X_raw, y_raw)

selected_features = [X_raw.columns[i] for i in np.argsort(rf.feature_importances_)[::-1][:TOP_FEATURES_COUNT]]
print(f"[+] Селектирани колони за инференс: {selected_features}")

X_reduced = X_raw[selected_features]
del X_raw, rf; gc.collect()

# ============================================
# 5. ENCODING & SCALING
# ============================================
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y_raw)
num_classes = len(label_encoder.classes_)
class_names = list(label_encoder.classes_)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_reduced).astype('float32')
del X_reduced; gc.collect()

# ============================================
# 6. СЪЗДАВАНЕ НА ВРЕМЕВИ ПРОЗОРЦИ (3D СЕКВЕНЦИИ)
# ============================================
print(f"\n[3] СЪЗДАВАНЕ НА ВРЕМЕВИ ПРОЗОРЦИ (Дължина: {SEQUENCE_LENGTH})...")
def create_sequences(X_data, y_data, seq_length):
    X_seq = np.zeros((len(X_data) - seq_length, seq_length, X_data.shape[1]), dtype='float32')
    y_seq = np.zeros(len(X_data) - seq_length, dtype='int64')
    for i in range(len(X_data) - seq_length):
        X_seq[i] = X_data[i:i+seq_length]
        y_seq[i] = y_data[i+seq_length-1]
    return X_seq, y_seq

X_seq, y_seq = create_sequences(X_scaled, y_encoded, SEQUENCE_LENGTH)
del X_scaled, y_encoded; gc.collect()

X_train, X_test, y_train, y_test = train_test_split(X_seq, y_seq, test_size=0.15, random_state=42, stratify=y_seq)
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.10, random_state=42, stratify=y_train)
del X_seq, y_seq; gc.collect()

# ============================================
# 7. ИНИЦИАЛИЗАЦИЯ НА LSTM МОДЕЛА
# ============================================
print("\n[4] СГЛОБЯВАНЕ НА LSTM НЕВРОННАТА МРЕЖА...")
keras.backend.clear_session()

model = models.Sequential([
    layers.Input(shape=(SEQUENCE_LENGTH, TOP_FEATURES_COUNT)),
    layers.LSTM(64, return_sequences=True, name="lstm_layer_1"),
    layers.BatchNormalization(name="bn_layer_1"),
    layers.Dropout(0.3, name="drop_layer_1"),
    
    layers.LSTM(32, return_sequences=False, name="lstm_layer_2"),
    layers.BatchNormalization(name="bn_layer_2"),
    layers.Dropout(0.2, name="drop_layer_2"),
    
    layers.Dense(num_classes, activation='softmax', name="output_dense")
])

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# ============================================
# 8. ОБУЧЕНИЕ (TRAINING)
# ============================================
print("\n[5] СТАРТИРАНЕ НА ОБУЧЕНИЕТО...")
early_stop = callbacks.EarlyStopping(monitor='val_loss', patience=4, restore_best_weights=True, verbose=1)
reduce_lr = callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=1e-6, verbose=1)

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    batch_size=1024,
    epochs=12,
    callbacks=[early_stop, reduce_lr],
    verbose=1
)

# ============================================
# 9. ОЦЕНКА И ГЕНЕРИРАНЕ НА ГРАФИКИ ЗА ДИПЛОМНАТА
# ============================================
print("\n" + "=" * 60)
print("[6] ОЦЕНКА НА ОБУЧЕНИЯ МОДЕЛ")
print("=" * 60)

loss, acc = model.evaluate(X_test, y_test, verbose=0)
y_pred = np.argmax(model.predict(X_test, verbose=0), axis=1)

print("\n--- Classification Report ---")
report_dict = classification_report(y_test, y_pred, target_names=class_names, zero_division=0, output_dict=True)
print(classification_report(y_test, y_pred, target_names=class_names, zero_division=0))

# ГРАФИКА 1: Матрица на объркването
plt.figure(figsize=(12, 10))
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
plt.title('Confusion Matrix', fontsize=14, fontweight='bold')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.savefig('chart_1_confusion_matrix.png', dpi=300)
plt.close()

# ГРАФИКА 2: F1-Score диаграма
f1_scores = [report_dict[cls]['f1-score'] for cls in class_names if cls in report_dict]
plt.figure(figsize=(12, 6))
plt.bar(class_names, f1_scores, color=sns.color_palette("viridis", len(class_names)))
plt.title('F1-Score per Class (Memory Safe Baseline)', fontsize=14, fontweight='bold')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('chart_2_f1_scores_bars.png', dpi=300)
plt.close()

print("[+] Графиките за презентацията бяха успешно записани!")

# ============================================
# 10. ЕКСПОРТ НА АРТЕФАКТИТЕ ЗА LIVE DEMO
# ============================================
print("\n[7] ЗАПИСВАНЕ НА ЧИСТИТЕ СУРОВИ МАТРИЦИ ЗА LIVE DEMO...")
model.save('cicids_intrusion_model.keras')
joblib.dump(scaler, 'scaler.pkl')
joblib.dump(label_encoder, 'label_encoder.pkl')
joblib.dump(selected_features, 'feature_names.pkl') 

clean_weights_dict = {layer.name: layer.get_weights() for layer in model.layers if len(layer.get_weights()) > 0}
joblib.dump(clean_weights_dict, 'cicids_numpy_weights.pkl')

print("\n" + "=" * 60)
print("ПЪЛНОТО ОБУЧЕНИЕ И ЕКСПОРТЪТ ПРИКЛЮЧИХА БЕЗУПРЕЧНО!")
print("=" * 60)

I0000 00:00:1780596307.359879    5824 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


NETWORK INTRUSION DETECTION - ULTIMATE LSTM TRAINING

[1] ЗАРЕЖДАНЕ НА ДАННИТЕ С ОН-ЛАЙН ФИЛТРИРАНЕ...
Обработка на: 03-02-2018.csv ...


W0000 00:00:1780596310.708264    5824 gpu_device.cc:2364] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


[✓] Успешно извлечени редове от 03-02-2018.csv: 4000
Обработка на: 02-20-2018.csv ...
[✓] Успешно извлечени редове от 02-20-2018.csv: 4000
Обработка на: 02-21-2018.csv ...
[✓] Успешно извлечени редове от 02-21-2018.csv: 5730
Обработка на: 02-14-2018.csv ...
[✓] Успешно извлечени редове от 02-14-2018.csv: 6000
Обработка на: 02-15-2018.csv ...
[✓] Успешно извлечени редове от 02-15-2018.csv: 6000
Обработка на: 02-22-2018.csv ...
[✓] Успешно извлечени редове от 02-22-2018.csv: 2362
Обработка на: 02-23-2018.csv ...
[✓] Успешно извлечени редове от 02-23-2018.csv: 2566
Обработка на: 02-28-2018.csv ...
[✓] Успешно извлечени редове от 02-28-2018.csv: 4000
Обработка на: 02-16-2018.csv ...
[✓] Успешно извлечени редове от 02-16-2018.csv: 6000
Обработка на: 03-01-2018.csv ...
[✓] Успешно извлечени редове от 03-01-2018.csv: 4000

[2] FEATURE SELECTION: Извличане на топ 15 характеристики...
[+] Селектирани колони за инференс: ['Init Fwd Win Byts', 'Fwd Seg Size Min', 'Fwd Header Len', 'Flow Duration'